# lift runner: Kaggle T4 x2 (backup GPU)

Reported runs use a Colab A100 in bf16 (`colab/README.md`). This notebook is the backup runner; `kaggle/README.md` has the one-time setup.

1. **Settings.** Accelerator *GPU T4 x2*, Internet *on*. Attach the Kaggle secrets `HF_TOKEN`, `GITHUB_TOKEN` and `LIFT_RUNS_REPO`, plus `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` and `LIFT_JUDGE_MAX_USD` for frontier-judge jobs.
2. **Cell 1, once per session.** Set `BRANCH`, then run it. It reads the secrets into `SECRETS` without printing them, clones the repo and installs it with `kaggle/bootstrap.sh`. The secrets never go into the environment: git, pip and uv never see them, and cell 2 gives them only to the `lift` commands.
3. **Cell 2, the job.** Paste the `lift` lines Claude Code printed into `JOB` and run the cell. Lines run in order and the first failing line stops the job. Outputs are pushed to the runs repo after every line and once more at the end, even after a failure or an interrupt. A push that exits 1 is retried (3 attempts over about 3 minutes), an exit 3 means conflicts whose local copies went under `conflicts/` in the runs repo, and a push never stops the job. The cell raises at the end if anything failed. For anything over 20 minutes, use **Save Version > Save & Run All (Commit)**.

**The T4 rule.** A T4 has no bf16, so every `generate`, `judge` and `run-dev` line takes `--dtype fp16`. A T4 number counts only if Gate 0's fp16-vs-bf16 check (`lift smoke --compare-dtype fp16`) passes, and fp16 numbers are never paired with bf16 ones. The generator runs on `cuda:0` and the judge on `cuda:1`.

**Privacy.** Kaggle saves `/kaggle/working` and the printed output with every notebook version. The clone, the venv and every data, output and validation file live under `/kaggle/temp/lift`; only `lift_env` (the interpreter path) goes in `/kaggle/working`. Kaggle describes `/kaggle/temp` as not saved, which is unverified here: check that the first committed version's Output holds only `lift_env`. Nothing here prints a secret or a model output. Keep this notebook private.

In [ ]:
import base64
import os
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

BRANCH = "main"
REPO_URL = "https://github.com/Knull/lost-in-finetuning.git"
# Kaggle saves /kaggle/working with every notebook version, so the clone, the venv and every data,
# output and validation file live under /kaggle/temp, which Kaggle describes as not saved (unverified).
ROOT = "/kaggle/temp/lift"
REPO = f"{ROOT}/repo"
LIFT_DIRS = {"LIFT_DATA_DIR": f"{ROOT}/data", "LIFT_OUTPUT_DIR": f"{ROOT}/outputs",
             "LIFT_VALIDATION_DIR": f"{ROOT}/validation"}
REQUIRED = ["HF_TOKEN", "GITHUB_TOKEN", "LIFT_RUNS_REPO"]
OPTIONAL = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "LIFT_JUDGE_MAX_USD"]
# No child process inherits these. The secrets go only to the lift commands (cell 2), and with Kaggle's
# client tokens any process could fetch every attached secret: pip and uv run third-party build code.
HIDDEN = [*REQUIRED, *OPTIONAL, "KAGGLE_USER_SECRETS_TOKEN", "KAGGLE_IAP_TOKEN"]


def load_secrets(get_secret):
    """Return the attached Kaggle secrets as a dict; they never go into os.environ (see HIDDEN).
    Unattached or empty optional ones are skipped, and no value is ever printed."""
    secrets = {}
    for name in REQUIRED + OPTIONAL:
        try:
            value = get_secret(name).strip()
        except Exception:
            continue
        if value:
            secrets[name] = value
    missing = [name for name in REQUIRED if name not in secrets]
    if missing:
        raise RuntimeError(f"attach these Kaggle secrets to the notebook (Add-ons > Secrets): {missing}")
    return secrets


def child_env(extra=()):
    """The environment of every child process: os.environ without the HIDDEN names, plus extra."""
    env = {name: value for name, value in os.environ.items() if name not in HIDDEN}
    env.update(extra)
    return env


def clone(token, run=subprocess.run):
    """Shallow-clone BRANCH into REPO. The token travels only in an HTTP header set through
    GIT_CONFIG_* variables: never in the URL, the command line, .git/config or an error message."""
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = child_env({"GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
                     "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                     "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"})
    shutil.rmtree(REPO, ignore_errors=True)
    result = run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO],
                 env=env, capture_output=True, text=True)
    if result.returncode != 0:
        detail = result.stderr.replace(token, "[redacted]").replace(auth, "[redacted]").strip()
        raise RuntimeError(f"git clone of {BRANCH!r} failed (exit {result.returncode}): {detail}")


SECRETS = load_secrets(UserSecretsClient().get_secret)
os.environ.update(LIFT_DIRS)
clone(SECRETS["GITHUB_TOKEN"])
if subprocess.run(["bash", f"{REPO}/kaggle/bootstrap.sh"], cwd=ROOT, env=child_env()).returncode != 0:
    raise RuntimeError("kaggle/bootstrap.sh failed: see its output above")

In [ ]:
import os
import shlex
import subprocess
import time

# One command per line, as Claude Code printed them; lines starting with # are skipped. There is no
# need to end with "lift sync push": the outputs are pushed after every line and once more at the end.
JOB = """
lift sync pull
lift info
# paste the stage's commands here, one per line (on a T4, generate/judge/run-dev take --dtype fp16)
"""
# lift sync push exits 0 when done, 1 when not everything was sent (safe to retry) and 3 when some files
# conflicted: their local copies went under conflicts/ in the runs repo, so nothing is lost.
PUSH_RETRY_WAITS = (60, 120)  # seconds before the second and the third attempt of a push that exits 1


def run_job(job, lift_py, *, cwd, secrets, run=subprocess.run, sleep=time.sleep):
    """Run each line in order and stop at the first failing line that is not a push. After every
    successful line except lift sync lines, push the outputs to the runs repo as a checkpoint, and push
    once more in finally, even after a failure or an interrupt. A push that exits 1 is retried; no push
    stops the job, and the next one sends whatever is still missing. Only the lift commands get the
    secrets. Raise at the end if any line or push did not succeed (an exit 3 included), so a committed
    version shows as failed."""
    problems = []

    def report(problem):
        problems.append(problem)
        print("!!", problem, flush=True)

    def call(line):
        cmd, extra = shlex.split(line), {}
        if cmd[0] == "lift":
            cmd, extra = [lift_py, "-m", "lift", *cmd[1:]], secrets
        print(">>", line, flush=True)
        return run(cmd, cwd=cwd, env=child_env({**extra, "PYTHONUNBUFFERED": "1"})).returncode

    def push(line, what):
        for attempt, wait in enumerate((*PUSH_RETRY_WAITS, None), start=1):
            rc = call(line)
            if rc != 1 or wait is None:
                break
            print(f"!! {what} exited 1 (attempt {attempt}); retrying in {wait} s", flush=True)
            sleep(wait)
        if rc == 3:
            report(f"{what} exited 3 (conflicts: the local copies are under conflicts/ in the runs repo)")
        elif rc == 1:
            report(f"{what} exited 1 on all {attempt} attempts")
        elif rc != 0:
            report(f"{what} exited {rc}")

    lines = [line.strip() for line in job.splitlines()]
    try:
        for line in lines:
            if not line or line.startswith("#"):
                continue
            words = shlex.split(line)
            if words[:3] == ["lift", "sync", "push"]:
                push(line, repr(line))
                continue
            rc = call(line)
            if rc != 0:
                report(f"{line!r} exited {rc}")
                break
            if words[:2] != ["lift", "sync"]:
                push("lift sync push", f"the checkpoint push after {line!r}")
    finally:
        push("lift sync push", "the final push")
    if problems:
        raise RuntimeError("job FAILED: " + "; ".join(problems))
    print("job OK: every line succeeded and the outputs are pushed", flush=True)


if "SECRETS" not in globals() or "LIFT_OUTPUT_DIR" not in os.environ:
    raise RuntimeError("run the setup cell (cell 1) first: a restarted kernel has lost the secrets and the "
                       "LIFT_* dirs")
with open("/kaggle/working/lift_env") as f:
    LIFT_PY = f.read().split("=", 1)[1].strip()
# Relative paths in JOB resolve against ROOT, which holds data/, outputs/ and validation/.
run_job(JOB, LIFT_PY, cwd=ROOT, secrets=SECRETS)